# Interactive Notebook 04 - PMSM basic control:

This interactive Jupyter notebook introduces basic control concepts for permanent magnet synchronous motors.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.io import loadmat
from pathlib import Path
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

**Throught the whole notebook, we will assume that the rotation speed $\omega$ can be set arbitrarily from an external load machine.**

### Control of the nonlinear PMSM:

We will use an adapted implementation from the simulation exercise as our plant to be controlled (code in `pmsm.py`).

Now we want to control the system to stay at a specific current operating point $\boldsymbol{i}^*_{\mathrm{s, dq}}$.

We are looking at the field-oriented current controller compenent of our complete control structure:

<div>
<img src="fig/full_controller.png" width="800"/>
</div>

<div>
<img src="fig/current_controller.png" width="800"/>
</div>

In [ ]:
# starting with this notebook, all auxiliary simulation code for the PMSM will be placed in 
# `CTPD/interactive_notebooks/utils/`
import sys

from path_helper import get_folder_path
sys.path.insert(1, str(get_folder_path()))

In [ ]:
from utils.pmsm import PMSM, lut_interpolate, clip_voltage
from utils.motor_params import MotorVariant
from utils.visualization import visualize_trajectories, visualize_trajectories_with_reference
from utils.signals import aprbs

This is the PMSM object, that carries all necessary parameters and can be used to simulate the motor's behavior.
For simplicity, we only consider the explicit Euler method with $T_\mathrm{s} = 10^{-4} \, \mathrm{s}$.

In [ ]:
pmsm = PMSM(
    saturated=True,
    motor_variant=MotorVariant.BRUSA,
    T_s=1e-4,
)
pmsm

The motor simulation uses an observation and a state.

The observation is a 6-element vector containing the currents $i_\mathrm{d}$, $i_\mathrm{q}$, the transformed motor angle $\cos(\varepsilon)$,  $\sin(\varepsilon)$, the electrical angular velocity  $\omega_\mathrm{el}$, and the current mechanical torque $T$.
The state is a more expressive structure that holds everything that is necessary for the simulation.
For this motor, the difference between the two is negligible. The distinction is only really necessary when parts of the state are not measureable (e.g., for an induction machine).

In [ ]:
pmsm.obs_description

In [ ]:
obs, state = pmsm.reset()

print(obs)
print(state)

#### Dummy controller:

First, we consider a dummy controller that always outputs $\boldsymbol{u}_\mathrm{dq} = \begin{bmatrix} 0 \, \mathrm{V} & 0 \, \mathrm{V} \end{bmatrix}$ to get a first look at the structure of a control loop.

Also note that the control delay is neglected here.

In [ ]:
def dummy_controller(i_dq):
    u_dq = jnp.array([0.0, 0.0])
    return u_dq

In [ ]:
# simulate controller plant interaction:
sequence_length = 10_000

obs, state = pmsm.reset() # puts the state back to the equilibrium point, zero-speed
obs, state = pmsm.set_speed(n=500, state=state)

i_dq = obs[:2]

i_dq_sequence = [i_dq]
u_dq_sequence = []

for i in range(sequence_length):
    u_dq = dummy_controller(i_dq)

    obs, state = pmsm.step(state, u_dq)
    i_dq_next = obs[:2]

    i_dq = i_dq_next
    
    i_dq_sequence.append(i_dq)
    u_dq_sequence.append(u_dq)

i_dq_sequence = jnp.array(i_dq_sequence)
u_dq_sequence = jnp.array(u_dq_sequence)

visualize_trajectories(
    i_dq_sequences=[i_dq_sequence],
    u_dq_sequences=[u_dq_sequence], 
    T_s_list=[pmsm.T_s],
    omegas=[state.physical_state.omega_el],
    ode=None,
    params=pmsm.env_properties.static_params,
)

#### FOC-PI:

Next we will implement the field-oriented current controller from the lecture.

In [ ]:
class FOCController:

    def __init__(self, static_params, lut_grids, lut_values, T_s, d, rho, use_ARW=False, K_r=1):
        """Gathers all static parameters for the controller."""
        
        self.T_s = T_s
        self.static_params = static_params
        self.lut_grids = lut_grids
        self.lut_values = lut_values
        self.d = d
        self.rho = rho

        # Equation (3.88)
        self.kappa = jnp.sqrt(1 - self.d**2) / d

        # Equation (3.94)
        self.P_lambda = jnp.exp(-2*self.rho)
        self.S_lambda = 2 * jnp.exp(-self.rho) * jnp.cos(self.kappa*self.rho)

        self.z_r = jnp.exp(-self.rho) * jnp.cos(self.kappa*self.rho)
        self.use_ARW = use_ARW
        if self.use_ARW:
            self.K_r = K_r

    def park_transformation_matrix(self, eps):
        cos = jnp.cos(eps)
        sin = jnp.sin(eps)
        T_p = jnp.column_stack((cos, -sin, sin, cos)).reshape(2, 2)
        return T_p
    
    def get_parameters_for_op(self, i_dq):
        """Extracts the momentary value for the current-dependent quantities from the LUTs."""
        
        p_d = {q: lut_interpolate(*self.lut_grids[q], self.lut_values[q], *i_dq) for q in self.lut_grids}
        L_diff = jnp.column_stack([p_d[q] for q in ["L_dd", "L_dq", "L_qd", "L_qq"]]).reshape(2, 2)
        psi_dq = jnp.column_stack([p_d[q] for q in ["Psi_d", "Psi_q"]])
        return L_diff, jnp.squeeze(psi_dq)
    
    def predict_currents(self, i_dq, u_dq, omega):
        """Equation (3.56)."""
        L_diff, psi_dq = self.get_parameters_for_op(i_dq)
        L_diff_inv = jnp.linalg.inv(L_diff)

        T_p_k = self.park_transformation_matrix(-omega * self.T_s)
       
        i_dq_pred = (
            (jnp.eye(2) - L_diff_inv @ T_p_k * self.T_s * self.static_params.r_s) @ i_dq 
            + L_diff_inv @ (T_p_k - jnp.eye(2)) @ psi_dq + L_diff_inv @ T_p_k * self.T_s @ u_dq
        )

        return i_dq_pred

    def compute_gains(self, L_diff):
        """Equations (3.71) for a and b, then Equation (3.95)."""
        L_dd = L_diff[0,0]
        L_qq = L_diff[1,1]
        
        a = jnp.array([
            1 - (self.T_s * self.static_params.r_s) / L_dd,
            1 - (self.T_s * self.static_params.r_s) / L_qq,
        ])

        b = jnp.array([
            self.T_s / L_dd,
            self.T_s / L_qq,
        ])

        K_p = (a - self.P_lambda) / b
        K_i = (1 - self.S_lambda + self.P_lambda) / (b * self.T_s)
        
        K_f = (self.z_r * K_i * self.T_s) / (1 - self.z_r) - K_p
        
        return K_p, K_i, K_f

    def integrate(self, integrated, e):
        """Equation (3.74)."""
        integrated = (integrated + e * self.T_s)
        return integrated
    
    def decouple(self, i_dq, v, L_diff, psi_dq, omega):
        """Equation (3.69)."""
        T_p_k = self.park_transformation_matrix(-omega * self.T_s)
        T_p_k_inv = jnp.linalg.inv(T_p_k)

        D_inv = jnp.array([
            [1 / L_diff[0,0], 0],
            [0, 1 / L_diff[1,1]]
        ])        
        u_dq = (
            (T_p_k_inv - jnp.eye(2)) / self.T_s @ psi_dq
            + (jnp.eye(2) - T_p_k_inv @ L_diff @ D_inv) * self.static_params.r_s @ i_dq
            + T_p_k_inv @ L_diff @ D_inv @ v
        )
        return u_dq

    def reset(self):
        return jnp.zeros((2,))


    def correct_integration(self, integrated, delta_u_dq, K_i, K_p):
        """Equations (3.118) and (3.113)."""
        K_aw = K_i * self.T_s / (K_p + K_i * self.T_s) * self.K_r
        
        return integrated - K_aw / K_i * delta_u_dq

    @eqx.filter_jit
    def __call__(self, i_dq, i_dq_ref, u_dq_appl, omega, eps, integrated):

        i_dq_pred = self.predict_currents(i_dq, u_dq_appl, omega)
        e = (i_dq_ref - i_dq_pred)

        integrated = self.integrate(integrated, e)
        L_diff, psi_dq = self.get_parameters_for_op(i_dq_pred)
        K_p, K_i, K_f = self.compute_gains(L_diff)

        # Equations (3.75), (3.76), and (3.73)
        v = K_p * e + K_i * integrated + K_f * i_dq_ref

        u_dq = self.decouple(i_dq_pred, v, L_diff, psi_dq, omega)
        u_dq_clipped = clip_voltage(u_dq, self.static_params.u_dc, eps, omega, self.T_s)

        if self.use_ARW:     
            delta_u_dq = u_dq - u_dq_clipped
            integrated = self.correct_integration(integrated, delta_u_dq, K_i, K_p)

        return u_dq_clipped, integrated

Now we also consider the control delay.
In a specfic timestep, the voltage command is computed that is to be applied in the next timestep.

In [ ]:
def simulate_controller_plant_interaction(pmsm, controller, i_dq_ref_sequence, n):
    obs, state = pmsm.reset() # puts the state back to the equilibrium point, zero-speed
    obs, state = pmsm.set_speed(n=n, state=state)

    integrated = controller.reset()
    i_dq = obs[:2]
    u_dq = jnp.array([0.0, 0.0])

    i_dq_sequence = [i_dq]
    u_dq_sequence = [u_dq]
    
    for i_dq_ref in i_dq_ref_sequence:
        u_dq_next, integrated = controller(
            i_dq,
            i_dq_ref,
            u_dq,
            state.physical_state.omega_el,
            state.physical_state.epsilon,
            integrated
        )
    
        obs, state = pmsm.step(state, u_dq)
        i_dq_next = obs[:2]
    
        i_dq = i_dq_next
        u_dq = u_dq_next
        
        i_dq_sequence.append(i_dq)
        u_dq_sequence.append(u_dq)
    
    i_dq_sequence = jnp.array(i_dq_sequence)
    u_dq_sequence = jnp.array(u_dq_sequence)

    final_state = state
    return i_dq_sequence, u_dq_sequence, final_state

In [ ]:
# compute reference
sequence_length = 1_000
i_dq_ref_sequence = aprbs(sequence_length, 2, t_min=50, t_max=300, key=jax.random.key(0)).T[0] * 200

# initialize controller
controller = FOCController(
    static_params=pmsm.env_properties.static_params,
    lut_grids=pmsm.LUT_grids,
    lut_values=pmsm.LUT_values,
    T_s=pmsm.T_s,
    d=0.99,
    rho=0.25,
) 

# run simulation
i_dq_sequence, u_dq_sequence, state = simulate_controller_plant_interaction(
    pmsm=pmsm,
    controller=controller,
    i_dq_ref_sequence=i_dq_ref_sequence,
    n=500,
)

# visualize results
visualize_trajectories_with_reference(
    i_dq_sequences=[i_dq_sequence],
    u_dq_sequences=[u_dq_sequence], 
    i_dq_ref_sequence=i_dq_ref_sequence,
    T_s_list=[pmsm.T_s],
    omegas=[state.physical_state.omega_el],
    ode=None,
    params=pmsm.env_properties.static_params,
)

#### Anti-reset wind-up (ARW)
While overshooting can generally occur, the overshoot that we can see for the previous controller is due to the integrator integrating the error further, while the voltage is already at the limit. 
With Anti-reset wind-up, the integration is modified with a correction feedback that reduces the integration when the voltage is at the limit.

In [ ]:
# compute reference
sequence_length = 1_000
i_dq_ref_sequence = aprbs(sequence_length, 2, t_min=50, t_max=300, key=jax.random.key(0)).T[0] * 200

i_dq_sequences = []
u_dq_sequences = []

# run twice, once without ARW once with ARW
for use_ARW in [False, True]:

    controller = FOCController(
        static_params=pmsm.env_properties.static_params,
        lut_grids=pmsm.LUT_grids,
        lut_values=pmsm.LUT_values,
        T_s=pmsm.T_s,
        d=0.99,
        rho=0.25,
        use_ARW=use_ARW,
    ) 
        
    i_dq_sequence, u_dq_sequence, state = simulate_controller_plant_interaction(
        pmsm=pmsm,
        controller=controller,
        i_dq_ref_sequence=i_dq_ref_sequence,
        n=500,
    )

    i_dq_sequences.append(i_dq_sequence)
    u_dq_sequences.append(u_dq_sequence)

visualize_trajectories_with_reference(
    i_dq_sequences=i_dq_sequences,
    u_dq_sequences=u_dq_sequences, 
    i_dq_ref_sequence=i_dq_ref_sequence,
    T_s_list=[pmsm.T_s, pmsm.T_s],
    labels=["without ARW", "with ARW"],
    omegas=[state.physical_state.omega_el, state.physical_state.omega_el],
    ode=None,
    params=pmsm.env_properties.static_params,
)

Now we can see that the controller with ARW does not overshoot as heavily and, therefore, does not exceed the voltage limit.